# Onderzoek: Hoe Agenttools Ondersteuning Bieden

> **Scope:** vergelijking van de meest gebruikte agent-frameworks (LangGraph, CrewAI, AutoGen / Microsoft Agent Framework) op zeven dimensies, met directe vertaling naar de **Boodschappen Agent** workflow (Node 1–9).

---

## 0. Frameworks in scope

| Framework | Orchestratiemodel | Status 2025–2026 | Sterkste punt |
|---|---|---|---|
| **LangGraph** | State-machine graphs (nodes + edges) | Actief; gebruikt door Klarna, Replit, Elastic [doc1] | Productie-grade observability + durable execution [doc1][doc2] |
| **CrewAI** | Role-based crews + Flows | Actief, v0.98+ [doc1] | Snel prototypen, lage boilerplate [doc1][doc3] |
| **AutoGen → Microsoft Agent Framework (MAF)** | Conversational + sinds okt 2025 ook graph-workflows | AutoGen in maintenance; MAF is opvolger (v1.0 GA april 2026) [doc1][doc10] | Combineert AutoGen's conversaties + Semantic Kernel enterprise features [doc8] |

> Microsoft heeft AutoGen en Semantic Kernel in oktober 2025 samengevoegd tot **Microsoft Agent Framework**; AutoGen ontvangt nog security-patches maar geen nieuwe features [doc1][doc8].

---

## 1. Workflows

**Wat het is:** de manier waarop een agent een meerstaps-taak structureert (lineair, parallel, conditioneel, cyclisch).

| Framework | Ondersteuning |
|---|---|
| **LangGraph** | Directed graphs met expliciete nodes + edges; ondersteunt **cyclische** workflows, conditional branching en parallelle node-uitvoering [doc1][doc4][doc5]. |
| **CrewAI** | Twee modellen: **Crews** (autonome teams) en **Flows** (event-driven, deterministische pipelines met expliciete state + branching) [doc3]. |
| **MAF / AutoGen** | MAF voegt **graph-based workflows** toe bovenop het conversatie-model, met type-safe routing, checkpointing en human-in-the-loop [doc8]. Sample-repo demonstreert Sequential, Concurrent en Conditional workflows [doc10]. |

**Toepassing op Boodschappen Agent:**
Onze 9-node flow met loops (4→2, 5→3, 6→4, 7→6, 8→6, 9→1) past het beste op een **graph-based** model. LangGraph of MAF-Workflows zijn dus structureel de natuurlijkste mapping; CrewAI Flows is een lichtere alternatief als we minder cyclische logica nodig hebben.

---

## 2. Processturing / Orchestration

**Wat het is:** wie bepaalt welke stap wanneer wordt uitgevoerd, en hoe agents/tools elkaar aanroepen.

| Framework | Orchestratiestijl |
|---|---|
| **LangGraph** | Deterministische edge-transitions op basis van state; ondersteunt parallel node execution (→ +40% snelheidswinst gerapporteerd op concurrente taken) [doc2]. |
| **CrewAI** | Sequential / hierarchical process; taken worden automatisch gedelegeerd tussen rollen [doc5]. Beperking: lage determinisme in pure crew-modus (rapporteerd 20% escalatie zonder validatie) [doc2]. |
| **MAF** | Combineert AutoGen's eenvoudige agent-abstracties met Semantic Kernel's enterprise-orchestration (sessions, middleware, filters, telemetrie) [doc8]. Patterns: sequential, concurrent, handoff [doc9]. |

**Toepassing:** Voor Node 6 (Vergelijking Supermarkt) en Node 7 (Route Planning) is **deterministische orchestratie** essentieel — dit pleit voor LangGraph of MAF-Workflows. Voor Node 1 (intent-extractie) en Node 8 (advies) kan een conversational/role-based agent prima.

---

## 3. State / Memory

**Wat het is:** hoe context tussen stappen en sessies behouden blijft.

| Framework | Memory-model |
|---|---|
| **LangGraph** | **Sterkst** in state management: typed state graphs, checkpointing per node, short-term (thread state) + long-term persistent memory, "time-travel" debugging via LangSmith [doc1][doc3][doc5]. Recovery na crash zonder context-verlies [doc2]. |
| **CrewAI** | Role-based memory in lagen: short-term, long-term, entity memory, contextual memory + RAG-ondersteuning per agent [doc3]. |
| **MAF** | Session-based state management, `AIContextProvider`, `ChatHistoryProvider` en `CompactionStrategy` voor lange contexten [doc7][doc8]. |

**Toepassing:** Onze **Vector DB** voor gebruikersprofiel, prijsgeschiedenis en geaccepteerde winkels mapt direct op:
- LangGraph: long-term store + state-checkpoint per node (vooral Node 9 → 1 transition).
- MAF: `AIContextProvider` voor profielinjectie vóór elke run.
- CrewAI: entity memory voor "winkel" en "product" entiteiten.

---

## 4. Informatie ophalen uit kennisbronnen (RAG)

**Wat het is:** semantische zoekacties in een vector-/document-store, of het aanroepen van externe knowledge-APIs.

| Framework | RAG-ondersteuning |
|---|---|
| **LangGraph** | Volledige toegang tot LangChain-ecosysteem: vector stores, hybrid search, retrievers, herrangschikkers [doc1][doc4]. |
| **CrewAI** | Ingebouwde RAG per agent-rol; integraties met o.a. LlamaIndex en SerperDev [doc3][doc4]. |
| **MAF** | `TextSearchProvider` als out-of-the-box RAG-context provider; modes: **BeforeAIInvoke** (search vóór elk model-call) of **on-demand via function calling** [doc6]. Integreert met VectorStore connectoren: Azure AI Search, Qdrant, Pinecone, Redis, Weaviate, in-memory [doc6]. Ook **GraphRAG** via Neo4j [doc6]. |

**Toepassing op Boodschappen Agent:**
- **Node 1:** profielgegevens injecteren vóór intent-extractie → MAF's `BeforeAIInvoke` mode is hier ideaal [doc6].
- **Node 5:** product-matching tegen winkel-SKU's via vector embeddings → hybrid search (keyword + semantic) [doc6].
- **Node 8:** persoonlijke voorkeuren ophalen voor toon van advies → on-demand RAG tool [doc6].

---

## 5. Feedbackloops

**Wat het is:** de mogelijkheid om terug te gaan in het proces (heroverweging), of menselijke feedback te verwerken.

| Framework | Loop / HITL-ondersteuning |
|---|---|
| **LangGraph** | Native cyclische workflows + **pause/resume hooks**; human-in-the-loop via checkpointing — exact resume vanuit zelfde state [doc1][doc3]. |
| **CrewAI** | Human checkpoints **per task** (`human_input=True`); Flows ondersteunen event-driven loops [doc3]. |
| **MAF / AutoGen** | UserProxyAgent kan op elk punt ingrijpen; MAF workflows ondersteunen native checkpointing en HITL [doc3][doc8]. |

**Toepassing op onze loops:**

| Loop | Aanbevolen mechanisme |
|---|---|
| Node 4 → 2 (geen winkels) | Conditional edge in graph (LangGraph/MAF) |
| Node 5 → 3 (onmatchbare producten) | Conditional edge met treshold-validator |
| Node 6 → 4 (onvoldoende dekking) | State-update + edge terug naar Node 4 |
| Node 8 → 6 (advies afgewezen) | Human-in-the-loop checkpoint; pause/resume |
| Node 9 → 1 (volgende sessie) | Persistent long-term memory write |

Verifier-pattern uit enterprise Agentic RAG-architectuur (claim-to-citation check) is direct toepasbaar op Node 8 om hallucinaties te voorkomen [doc9].

---

## 6. Tool Calling

**Wat het is:** het laten aanroepen van externe APIs/functies door de agent.

| Framework | Tool-calling |
|---|---|
| **LangGraph** | Volledig LangChain-toolecosysteem (honderden integraties); custom Python-functies als nodes [doc1][doc4]. |
| **CrewAI** | 100+ pre-built integraties (Gmail, Slack, Salesforce, HubSpot), CodeInterpreterTool [doc1][doc3]. |
| **MAF** | Native tool-calling, MCP (Model Context Protocol) server-integratie, function calling, code interpreter, Bing grounding, file search [doc8][doc10]. |

**Toepassing op Boodschappen Agent — tools per node:**

| Tool | Gebruikt in Node | Framework-mapping |
|---|---|---|
| **Google Maps Geocoding API** | 2 | Custom function tool in elk framework |
| **Google Maps Places API** | 4 | Idem |
| **Google Maps Distance Matrix / Directions API** | 4, 7 | Idem |
| **`bartmachielsen/supermarktconnector`** (Python lib) | 5 | Wrap als custom tool (LangGraph node / CrewAI tool / MAF AIFunction) |
| **Optioneel: Spoonacular recepten-API** | 3 | Idem |

Beste practices uit MAF: scheid **Planner** en **Executor** agents zodat tool-aanroepen apart auditbaar zijn; gebruik idempotency keys en dry-run mode voor high-risk tools [doc9].

---

## 7. Validatie

**Wat het is:** zekerheid dat de agent correcte output produceert, juiste tools aanroept, en geen hallucineert.

### Built-in framework validatie

| Framework | Validatie-mechanismes |
|---|---|
| **LangGraph** | State-enforced structured output (94% accuratesse op task completion benchmarks) [doc2]; LangSmith voor tracing + evals [doc1]. |
| **CrewAI** | Role-enforced output (91% accuratesse maar lager determinisme — vereist extra validatie tussen agents) [doc2]. |
| **MAF / AutoGen** | Flexibele output (89% accuratesse) [doc2]; middleware voor intercepting agent acties + telemetrie via OpenTelemetry [doc8][doc10]. |

### Externe evaluatie-tools

- **AgentEval** (.NET, voor MAF): fluent assertions voor tool-chains, stochastische evaluatie (mean/stddev/success-rate i.p.v. pass/fail), model-comparison, RAG-metrics (faithfulness, relevance, context precision), red-team probes (192 probes, OWASP LLM Top 10), responsible-AI metrics (toxiciteit, bias) [doc7].
- **RAGAS / DeepEval** voor Python-equivalent [doc7].
- **LangSmith** voor LangGraph debugging en evals [doc1][doc5].

### Validatie-checklist per Boodschappen-node

| Node | Validatie | Mechanisme |
|---|---|---|
| 1 | Schema-check op intent-JSON | Pydantic / JSON-schema validator |
| 2 | Postcode-format, NL bounding box | Deterministische rule node |
| 3 | Allergie/dieet-conflict | Rule check + LLM-as-judge |
| 4 | ≥1 winkel binnen radius + open | Deterministische conditie |
| 5 | Prijs > 0 en < plafond; eenheid-match | Anti-outlier rule + embedding-similarity |
| 6 | Dekking ≥ 80%, budget ≤ max | Rule + LLM-vergelijking voor uitleg |
| 7 | Reistijd binnen acceptabel | Rule |
| 8 | **Anti-hallucinatie**: elke prijs/winkel in tekst moet in input zitten | Faithfulness-metric (RAGAS/AgentEval) + verifier-agent [doc7][doc9] |
| 9 | Geen PII in embeddings | Pattern + LLM-check (ToxicityMetric / PII probe) [doc7] |

### LLM-vergelijking (Node 6 & 8)

Stochastische evaluatie aanbevolen: niet één run, maar **N runs** met success-rate threshold. Vergelijk GPT-4o vs Claude vs Gemini op:
- mean score + standaarddeviatie (consistentie)
- tool-accuracy
- faithfulness (geen hallucinatie)
- latency + cost per 1K requests [doc7]

Voorbeeldoutput van model-comparer geeft een directe winner + best-value aanbeveling [doc7].

---

## Conclusie & Aanbeveling voor Boodschappen Agent

| Criterium | Voorkeur |
|---|---|
| **Architectuur** | LangGraph **of** Microsoft Agent Framework Workflows — beide ondersteunen graph + loops + checkpointing native. |
| **Productie-ready** | LangGraph (Klarna, Replit, Elastic in productie) [doc1] of MAF (v1.0 GA april 2026, enterprise support) [doc10]. |
| **Snel prototype eerst** | CrewAI voor MVP, daarna migreren [doc5]. |
| **RAG (Vector DB)** | MAF `TextSearchProvider` + Azure AI Search of Qdrant [doc6]; LangChain retrievers in LangGraph. |
| **Tool integratie (Maps + supermarktconnector)** | Custom function tools — alle drie ondersteunen dit; MAF heeft daarnaast MCP voor herbruikbare server-tools [doc8]. |
| **Validatie / LLM-vergelijking** | AgentEval (voor MAF/.NET) of RAGAS (Python) voor stochastische eval + faithfulness [doc7]. |
| **Loops (terugstappen)** | LangGraph conditional edges; MAF workflow checkpointing met pause/resume [doc1][doc8]. |

**Praktisch advies:** start met een **hybride aanpak** — LangGraph (of MAF) als orchestratie-backbone met deterministische nodes voor regel-gebaseerde stappen (2, 4, 6, 7), en LLM-agents voor de creatieve nodes (1, 3, 8). Combineer dit met een Verifier-agent vóór Node 9 om hallucinaties uit het advies te filteren [doc9].

---

## Bronnen

- [doc1] JetThoughts — *LangGraph vs CrewAI vs AutoGen: Open Source Alternatives 2025*
- [doc2] Braincuber — *CrewAI vs AutoGen vs LangGraph: Framework Comparison 2025/2026*
- [doc3] DataCamp — *CrewAI vs LangGraph vs AutoGen* (Sep 2025)
- [doc4] Latenode — *LangGraph vs AutoGen vs CrewAI Complete Comparison 2025*
- [doc5] Agenticai-flow — *AI Agent Framework Comparison*
- [doc6] Microsoft Learn — *Agent Framework: RAG*
- [doc7] AgentEvalHQ/AgentEval — GitHub repository
- [doc8] Microsoft Learn — *Microsoft Agent Framework Overview*
- [doc9] Chetan Kerhalkar (Medium) — *Advanced Agentic RAG with Microsoft Agent Framework*
- [doc10] microsoft/Agent-Framework-Samples — GitHub repository
